In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

# Feature Extraction
an approach to data compression: improving storage and computational efficiency, reducing the curse of dimensionality
### Principal Component Analysis (PCA)
- identify patterns based on correlation between features to find orthogonal directions of highest variance
- project the data onto new subspace w <= dimenstions
- construct $d \times k$-dimensional transformation matrix $\mathbf W$
$$
\mathbf x = [x_1, x_2, ..., x_d],  \mathbf x \in \mathbb R^d \\
\mathbf{xW} = \mathbf z \\
\mathbf z = [z_1, z_2, ..., z_k], \mathbf z \in \mathbb R^k
$$
\
Applications: EDA, denoising stock market signals, genome data and gene expression analysis in bioinformatics \
summarized approach: 
1. standardize the dataset
2. construct covariance matrix
3. decompose into eigenvectors and eigenvalues
4. sort eigenvalues in decreasing order to rank eigenvectors
5. select top k eigenvectors
6. projection matrix
7. transform the dataset

In [ ]:
import pandas as pd
df_wine = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data',
    header=None
)

In [ ]:
from sklearn.model_selection import train_test_split
X, y = df_wine.iloc[:, 1:].values, df_wine.iloc[:, 0].values
X_train, X_test, y_train, y_test = \
    train_test_split(X, y, test_size=0.3, stratify=y,
                     random_state=0)

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train_std = sc.fit_transform(X_train)
X_test_std = sc.transform(X_test)

Covariance between two features $x_i$ and $x_j$:
$$
\sigma_{jk} = \frac{1}{n-1} \sum_{i=1}^n (x_j^{(i)} - \mu_j)(x_k^{(i)} - \mu_k)
$$
(means are zero for standardized dataset)
$$
\implies \sigma_{jk} = \frac{1}{n-1} \sum_{i=1}^n x_j^{(i)} x_k^{(i)}
$$

In [ ]:
import numpy as np
cov_mat = np.cov(X_train_std.T)
eigenvals, eigenvecs = np.linalg.eig(cov_mat)
eigenvals

$$\text{explained variance ratio} = \frac{\lambda_j}{\sum_{j=1}^d \lambda_j}$$

In [ ]:
tot = sum(eigenvals)
var_exp = [(i/tot) for i in sorted(eigenvals, reverse=True)]
cum_var_exp = np.cumsum(var_exp)

import matplotlib.pyplot as plt
plt.bar(range(1,14), var_exp, align='center',
        label='individual explained variance')
plt.step(range(1,14), cum_var_exp, where='mid',
         label='cumulative explained variance')
plt.ylabel('explained variance ratio')
plt.xlabel('principal component index')
plt.legend(loc='center right')
plt.tight_layout()
plt.show()

In [ ]:
eigen_pairs = [(np.abs(eigenvals[i]), eigenvecs[:, i]) for i in range(len(eigenvals))]
eigen_pairs.sort(key=lambda k: k[0], reverse=True)
w = np.hstack((eigen_pairs[0][1][:, np.newaxis],
               eigen_pairs[1][1][:, np.newaxis]))
w

In [ ]:
X_train_pca = X_train_std.dot(w)
colors = ['r', 'b', 'g']
markers = ['o', 's', '^']
for l, c, m in zip(np.unique(y_train), colors, markers):
    plt.scatter(X_train_pca[y_train==l, 0],
                X_train_pca[y_train==l, 1],
                c=c, label=f'class {l}', marker=m)
plt.xlabel('PC 1')
plt.ylabel('PC 2')
plt.legend(loc='best')
plt.tight_layout()
plt.show()

PCA w/ scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.decomposition import PCA
from utils import plot_decision_regions
pca = PCA(n_components=2)
lr = OneVsRestClassifier(LogisticRegression(solver='lbfgs',
                                            random_state=1))

X_train_pca = pca.fit_transform(X_train_std)
X_test_pca = pca.transform(X_test_std)
lr.fit(X_train_pca, y_train)
plot_decision_regions(X=X_train_pca, y=y_train, classifier=lr)
plt.xlabel('pc1')
plt.ylabel('pc2')
plt.legend(loc='best')
plt.tight_layout()
plt.show()